In [21]:
#Importing required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize']=(12,4)

In [23]:
#Reading in clinical datasets
history = pd.read_csv('../data/raw/clinical_history.csv')
labs =  pd.read_csv('../data/raw/laboratory_results.csv', parse_dates=['timestamp'])
patients =   pd.read_csv('../data/raw/patients.csv', parse_dates=['registration_date'])
outcome =  pd.read_csv('../data/raw/sepsis_outcomes.csv', parse_dates=['diagnosis_time'])
vitals =  pd.read_csv('../data/raw/vital_signs.csv', parse_dates=['timestamp'])

tables = {'history': history, 'patients': patients, 'labs': labs, 'outcome':outcome, 'vitals' :vitals}

for name, df in tables.items():
    print(f'{name:10s}. Shape={df.shape}')

history   . Shape=(1449, 6)
patients  . Shape=(600, 5)
labs      . Shape=(2430, 8)
outcome   . Shape=(600, 6)
vitals    . Shape=(11807, 8)


### Data Quality Assessment

In [24]:
#Checking for the percentage of missing values, duplicates, and summary statistics
for name, df in tables.items():
    print(f'{name.upper()}')
    print(f'==Total missing values:\n {df.isna().mean().mul(100).round(2)}')
    print(f'==Number of duplicates : {df.duplicated().sum()}')
    print(f'==Summary Statistics:\n{df.describe()}') 
    print('\n' + '='*30 + '\n' )

HISTORY
==Total missing values:
 history_id             0.00
patient_id             0.00
diagnosis_history      2.62
infection_history      0.00
medication_history     9.45
treatment_history     14.56
dtype: float64
==Number of duplicates : 0
==Summary Statistics:
        history_id   patient_id
count  1449.000000  1449.000000
mean    725.000000   303.839199
std     418.434583   172.910492
min       1.000000     1.000000
25%     363.000000   154.000000
50%     725.000000   302.000000
75%    1087.000000   455.000000
max    1449.000000   600.000000


PATIENTS
==Total missing values:
 patient_id             0.00
age                    0.00
gender                 0.00
medical_conditions    25.83
registration_date      0.00
dtype: float64
==Number of duplicates : 0
==Summary Statistics:
       patient_id         age    registration_date
count  600.000000  600.000000                  600
mean   300.500000   59.908333  2024-12-31 16:14:24
min      1.000000   18.000000  2024-01-01 00:00:00
25%

In [25]:
#Sepsis prevalence
print('Sepsis prevalence', outcome.sepsis_event.mean().round(2))
print(outcome.sepsis_event.value_counts())

Sepsis prevalence 0.12
sepsis_event
False    528
True      72
Name: count, dtype: int64


In [27]:
#Referential integrity
valid_ids = set(patients['patient_id'])
for name, df in [('history', history), ('labs', labs),\
                 ('outcome',outcome), ('vitals',vitals)]:
    orphans = (~df['patient_id'].isin(valid_ids)).sum()
    print(name, 'orphan_patient_id rows:', orphans)
    

history orphan_patient_id rows: 0
labs orphan_patient_id rows: 0
outcome orphan_patient_id rows: 0
vitals orphan_patient_id rows: 0


### Data Cleaning

In [ ]:
labs_cols = ['white_cell_count', 'crp',
            'lactate', 'creatinine', 'platelet_count']

In [ ]:
vital_cols = ['heart_rate','temperature', 'oxygen_saturation',
               'respiratory_rate','blood_pressure']

vitals[vital_cols] = vitals.groupby('patient_id')[vital_cols].transform(lambda x: x.ffill())
vitals[vital_cols] = vitals.groupby('patient_id')[vital_cols].fillna(vitals[vital_cols].median())

labs[labs_cols] = labs.groupby('patient_id')[labs_cols].transform(lambda x: x.ffill())
labs[labs_cols] = labs.groupby('patient_id')[labs_cols].fillna(labs[labs_cols].median())

In [40]:
print('Remaining missing vitals :', vitals.isna().sum().sum())
print('Remaining missing labs:', labs.isna().sum().sum())

Remaining missing vitals : 0
Remaining missing labs: 0


In [41]:
history.to_csv('../data/processed/history_clean.csv', index=False)
labs.to_csv('../data/processed/laboratory_clean.csv', index=False)
patients.to_csv('../data/processed/patients_clean.csv', index=False)
outcome.to_csv('../data/processed/outcomes_clean.csv', index=False)
vitals.to_csv('../data/processed/vitals_clean.csv', index=False)

### EDA